# N143 · 离线查询、扫描线与Mo算法

**目标：** 在不改变查询语义的前提下重排计算顺序。

**先修：** N036, N057, N083, N135。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 查询编号；按阈值排序；离线DSU；Fenwick；Mo增删；平方分块。

## 从问题到算法

离线允许先知道所有查询并改变处理顺序，最后按原查询ID恢复结果。最短覆盖区间按左端点加入候选堆；限边权连通性按阈值逐步union；Mo算法则让查询左右端点缓慢移动，以可逆增删维护窗口统计。离线算法不适用于必须立即返回、且未来查询未知的接口。

## 最小实现

**本章接口：** `min_interval(intervals, queries)`、`distance_limited_paths_exist(n, edges, queries)`、`mo_distinct_counts(nums, queries)`

In [1]:
class DSU:
    def __init__(self,n): self.parent=list(range(n)); self.size=[1]*n; self.components=n
    def find(self,x):
        while self.parent[x]!=x:
            self.parent[x]=self.parent[self.parent[x]]; x=self.parent[x]
        return x
    def union(self,a,b):
        a=self.find(a); b=self.find(b)
        if a==b: return False
        if self.size[a]<self.size[b]: a,b=b,a
        self.parent[b]=a; self.size[a]+=self.size[b]; self.components-=1
        return True

from heapq import heappush,heappop
from math import isqrt

def min_interval(intervals,queries):
    intervals=sorted(intervals); heap=[]; index=0; answer=[-1]*len(queries)
    for q,original in sorted((q,i) for i,q in enumerate(queries)):
        while index<len(intervals) and intervals[index][0]<=q:
            l,r=intervals[index]; heappush(heap,(r-l+1,r)); index+=1
        while heap and heap[0][1]<q: heappop(heap)
        if heap: answer[original]=heap[0][0]
    return answer

def distance_limited_paths_exist(n,edges,queries):
    edges=sorted(edges,key=lambda e:e[2]); ordered=sorted(enumerate(queries),key=lambda x:x[1][2]); dsu=DSU(n); index=0; answer=[False]*len(queries)
    for original,(u,v,limit) in ordered:
        while index<len(edges) and edges[index][2]<limit:
            a,b,w=edges[index]; dsu.union(a,b); index+=1
        answer[original]=dsu.find(u)==dsu.find(v)
    return answer

def mo_distinct_counts(nums,queries):
    n=len(nums); block=max(1,isqrt(n)); answer=[0]*len(queries)
    if any(not 0<=l<=r<=n for l,r in queries): raise IndexError('half-open query outside array')
    order=sorted(range(len(queries)),key=lambda i:(queries[i][0]//block,queries[i][1] if queries[i][0]//block%2==0 else -queries[i][1]))
    counts={}; left=right=0; distinct=0
    def add(i):
        nonlocal distinct
        x=nums[i]; counts[x]=counts.get(x,0)+1
        if counts[x]==1: distinct+=1
    def remove(i):
        nonlocal distinct
        x=nums[i]; counts[x]-=1
        if counts[x]==0: distinct-=1; del counts[x]
    for i in order:
        l,r=queries[i]
        while left>l: left-=1; add(left)
        while right<r: add(right); right+=1
        while left<l: remove(left); left+=1
        while right>r: right-=1; remove(right)
        answer[i]=distinct
    return answer

## 正确性、前提与复杂度

扫描到阈值q时，数据结构恰好包含符合阈值的对象。堆中允许保留非顶端过期区间，只要每次读答案前清除堆顶过期项。Mo每次增删准确更新元素频次与distinct，处理顺序只影响代价，不改变目标窗口统计。

**代价：** 区间与限边权查询O((N+Q)log(N+Q))级排序加堆/DSU；Mo O((N+Q)√N+QlogQ)平均字典操作，空间O(N+Q)。区间覆盖端点闭合；限边权严格小于limit；Mo内部统一半开区间。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

nums=[1,2,1,3,2,4]; queries=[(2,6),(0,3),(1,5),(0,3),(4,4)]; answers=mo_distinct_counts(nums,queries)
show_table(['原查询ID','半开区间','不同数个数'],[(i,q,answers[i]) for i,q in enumerate(queries)])

原查询ID,半开区间,不同数个数
0,"(2, 6)",4
1,"(0, 3)",2
2,"(1, 5)",3
3,"(0, 3)",2
4,"(4, 4)",0


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert min_interval([[1,4],[2,4],[3,6],[4,4]],[2,3,4,5])==[3,3,1,4]
assert distance_limited_paths_exist(2,[(0,1,2)],[(0,1,2),(0,1,3)])==[False,True]
assert mo_distinct_counts([],[(0,0)])==[0]
from random import Random
from collections import deque
rng=Random(143)
for _ in range(100):
    nums=[rng.randrange(7) for _ in range(20)]; queries=[]
    for _ in range(25):
        l=rng.randrange(21); r=rng.randrange(l,21); queries.append((l,r))
    assert mo_distinct_counts(nums,queries)==[len(set(nums[l:r])) for l,r in queries]
    intervals=[sorted(rng.sample(range(15),2)) for _ in range(8)]; qs=[rng.randrange(15) for _ in range(12)]
    assert min_interval(intervals,qs)==[min((r-l+1 for l,r in intervals if l<=q<=r),default=-1) for q in qs]
for _ in range(80):
    n=6; edges=[(rng.randrange(n),rng.randrange(n),rng.randrange(8)) for _ in range(12)]; queries=[(rng.randrange(n),rng.randrange(n),rng.randrange(9)) for _ in range(12)]; ref=[]
    for s,t,limit in queries:
        seen={s}; q=deque([s])
        while q:
            u=q.popleft()
            for a,b,w in edges:
                if w>=limit: continue
                v=b if a==u else a if b==u else None
                if v is not None and v not in seen: seen.add(v); q.append(v)
        ref.append(t in seen)
    assert distance_limited_paths_exist(n,edges,queries)==ref
print('N143: 所有本章断言通过')

N143: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 证明阈值严格小于与小于等于会改变结果。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 实现自编离线区间去重计数。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1851. Minimum Interval to Include Each Query（canonical）
- 1697. Checking Existence of Edge Length Limited Paths（canonical）
- 315. Count of Smaller Numbers After Self（comparison）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。